# Amazon Reviews 2023 — Smart subset for Multimodal Recommendation

Notebook này tạo dataset nghiên cứu từ `Clothing_Shoes_and_Jewelry` trên Kaggle mà **không cắt random theo từng dòng**. Quy trình:

1. Chỉ giữ interaction dương (`rating >= 4`) và khử trùng theo `(user_id, parent_asin)`, ưu tiên lần tương tác mới nhất.
2. Chọn user theo các dải mức hoạt động để không chỉ giữ heavy users. Trong từng dải, dùng stable hash để chọn xác định và tái lập; đây là lấy mẫu đại diện theo tầng, không phải random sampling.
3. Giữ **toàn bộ lịch sử dương** của user đã chọn, sau đó chạy iterative K-core trên đồ thị user–item.
4. Nếu vượt ngân sách, giảm theo user (không cắt interaction rời rạc), tiếp tục bảo toàn activity strata và chạy lại K-core.
5. Join metadata bằng `parent_asin`, loại item không có metadata thiết yếu, rồi split chronological leave-last-out.

> Bật Internet trong Kaggle nếu dùng URL. Tốt hơn: Add Data hai file `.jsonl.gz` vào `/kaggle/input`; notebook sẽ tự tìm file.

In [ ]:
# Kaggle thường đã có psutil; DuckDB được cài/upgrade nhẹ và xử lý out-of-core.
%pip install -q "duckdb>=1.1,<2" psutil


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, time
import duckdb, psutil

# ===== Tham số duy nhất thường cần chỉnh =====
TARGET_USERS = 40_000
TARGET_INTERACTIONS = 400_000
MIN_USER_DEGREE = 5       # đủ N-2 / valid / test và lịch sử có ý nghĩa
MIN_ITEM_DEGREE = 5
POSITIVE_RATING = 4.0
SELECTION_SEED = 20260813
MAX_ITERATIONS = 20
RESERVE_RAM_GB = 2.0
MAX_DUCKDB_RAM_GB = 6.0   # chỉ là trần, không phải RAM được giả định là có
THREADS = 2               # ít thread giúp giảm memory spike trên Kaggle

WORK = Path('/kaggle/working/datn_dataset')
WORK.mkdir(parents=True, exist_ok=True)
DB_PATH = WORK / 'prepare.duckdb'

REVIEW_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Clothing_Shoes_and_Jewelry.jsonl.gz'
META_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Clothing_Shoes_and_Jewelry.jsonl.gz'

vm = psutil.virtual_memory()
available_gb = vm.available / 1024**3
budget_gb = min(MAX_DUCKDB_RAM_GB, max(0.5, (available_gb - RESERVE_RAM_GB) * 0.55))
if available_gb <= RESERVE_RAM_GB + 0.5:
    raise MemoryError(f'Chỉ còn {available_gb:.2f} GiB RAM; hãy restart session trước khi chạy.')
print(f'RAM total={vm.total/1024**3:.2f} GiB, available={available_gb:.2f} GiB, DuckDB limit={budget_gb:.2f} GiB')


In [ ]:
def find_input(patterns):
    files = [p for pattern in patterns for p in Path('/kaggle/input').rglob(pattern)]
    return max(files, key=lambda p: p.stat().st_size) if files else None

def download_if_missing(found, url, filename):
    if found:
        print('Dùng Kaggle input:', found)
        return found
    target = WORK / filename
    if not target.exists():
        free = shutil.disk_usage(WORK).free / 1024**3
        print(f'Không thấy Kaggle input; còn {free:.1f} GiB disk. Đang tải {filename} ...')
        subprocess.run(['wget', '-c', '--progress=dot:giga', '-O', str(target), url], check=True)
    return target

review_path = download_if_missing(
    find_input(['Clothing_Shoes_and_Jewelry.jsonl.gz']), REVIEW_URL,
    'Clothing_Shoes_and_Jewelry.jsonl.gz')
meta_path = download_if_missing(
    find_input(['meta_Clothing_Shoes_and_Jewelry.jsonl.gz']), META_URL,
    'meta_Clothing_Shoes_and_Jewelry.jsonl.gz')
print('Reviews:', review_path, '\nMetadata:', meta_path)


## 1. Ingest tối thiểu, out-of-core

Không lưu review text ở dataset huấn luyện: giai đoạn recommender chỉ cần interaction. Review text có thể trích riêng sau khi item đã được chốt. Điều này giảm mạnh disk/RAM.

In [ ]:
con = duckdb.connect(str(DB_PATH))
con.execute(f"SET memory_limit='{budget_gb:.2f}GB'")
con.execute(f'SET threads={THREADS}')
con.execute(f"SET temp_directory='{(WORK / 'duckdb_tmp').as_posix()}'")
con.execute('SET preserve_insertion_order=false')

rp = review_path.as_posix().replace("'", "''")
con.execute(f"""
CREATE OR REPLACE TABLE positive AS
SELECT user_id::VARCHAR AS user_id, parent_asin::VARCHAR AS item_id,
       rating::FLOAT AS rating, timestamp::BIGINT AS timestamp,
       verified_purchase::BOOLEAN AS verified_purchase
FROM read_json_auto('{rp}', format='newline_delimited', compression='gzip',
                    columns={{user_id:'VARCHAR', parent_asin:'VARCHAR', rating:'DOUBLE',
                             timestamp:'BIGINT', verified_purchase:'BOOLEAN'}},
                    maximum_object_size=16777216, ignore_errors=true)
WHERE rating >= {POSITIVE_RATING} AND user_id IS NOT NULL AND parent_asin IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY user_id, parent_asin ORDER BY timestamp DESC) = 1
""")
print(con.execute('SELECT count(*) rows, count(DISTINCT user_id) users, count(DISTINCT item_id) items FROM positive').fetchdf())


## 2. Chọn user đại diện theo tầng hoạt động

Ba tầng `5–9`, `10–19`, `20+` nhận quota theo quy mô thật nhưng mỗi tầng được giữ ít nhất 15% nếu có đủ user. Stable hash chỉ làm tie-break tái lập, không dùng RNG và không cắt từng interaction.

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE user_stats AS
SELECT user_id, count(*) degree, min(timestamp) first_ts, max(timestamp) last_ts,
       CASE WHEN count(*) BETWEEN {MIN_USER_DEGREE} AND 9 THEN '05_09'
            WHEN count(*) BETWEEN 10 AND 19 THEN '10_19' ELSE '20_plus' END activity_band
FROM positive GROUP BY user_id HAVING count(*) >= {MIN_USER_DEGREE}
""")
bands = con.execute('SELECT activity_band, count(*) n FROM user_stats GROUP BY 1 ORDER BY 1').fetchall()
eligible = sum(n for _, n in bands)
floor = int(TARGET_USERS * 0.15)
quota = {band: min(n, max(floor, round(TARGET_USERS*n/eligible))) for band, n in bands}
# Sửa sai số quota, ưu tiên tầng còn nhiều capacity nhất.
while sum(quota.values()) > TARGET_USERS:
    b = max((b for b in quota if quota[b] > min(floor, dict(bands)[b])), key=lambda x: quota[x], default=None)
    if b is None: break
    quota[b] -= 1
while sum(quota.values()) < min(TARGET_USERS, eligible):
    b = max((b for b, n in bands if quota[b] < n), key=lambda x: dict(bands)[x]-quota[x], default=None)
    if b is None: break
    quota[b] += 1
print('Band sizes:', bands, 'Quota:', quota)
values = ','.join(f"('{b}',{q})" for b, q in quota.items())
con.execute(f"""
CREATE OR REPLACE TABLE selected_users AS
WITH quota(activity_band, n) AS (VALUES {values}), ranked AS (
  SELECT s.*, row_number() OVER (PARTITION BY s.activity_band
    ORDER BY hash(s.user_id || '{SELECTION_SEED}')) rn
  FROM user_stats s
) SELECT r.user_id, r.activity_band FROM ranked r JOIN quota q USING(activity_band) WHERE rn <= q.n
""")
con.execute('CREATE OR REPLACE TABLE core AS SELECT p.* FROM positive p JOIN selected_users u USING(user_id)')


## 3. Iterative K-core và giới hạn ngân sách

K-core phải lặp đến hội tụ vì loại item yếu có thể khiến user yếu và ngược lại. Nếu vẫn quá lớn, notebook xếp user bằng activity band + stable hash, giữ trọn history rồi chạy K-core lại.

In [ ]:
def iterative_kcore(table='core'):
    previous = -1
    for iteration in range(1, MAX_ITERATIONS + 1):
        current = con.execute(f'SELECT count(*) FROM {table}').fetchone()[0]
        if current == previous:
            print(f'K-core hội tụ sau {iteration-1} vòng: {current:,} rows')
            return
        previous = current
        con.execute(f"""
        CREATE OR REPLACE TABLE core_next AS
        WITH good_users AS (SELECT user_id FROM {table} GROUP BY 1 HAVING count(*) >= {MIN_USER_DEGREE}),
             good_items AS (SELECT item_id FROM {table} GROUP BY 1 HAVING count(*) >= {MIN_ITEM_DEGREE})
        SELECT c.* FROM {table} c JOIN good_users USING(user_id) JOIN good_items USING(item_id)
        """)
        con.execute(f'DROP TABLE {table}')
        con.execute(f'ALTER TABLE core_next RENAME TO {table}')
    raise RuntimeError('K-core chưa hội tụ; tăng MAX_ITERATIONS và chạy lại cell')

iterative_kcore()
rows = con.execute('SELECT count(*) FROM core').fetchone()[0]
if rows > TARGET_INTERACTIONS:
    # Giữ trọn user tới ngân sách interaction; các activity band được xen kẽ.
    con.execute(f"""
    CREATE OR REPLACE TABLE budget_users AS
    WITH band_ranked AS (
      SELECT c.user_id, u.activity_band, count(*) degree,
             row_number() OVER (PARTITION BY u.activity_band
               ORDER BY hash(c.user_id || '{SELECTION_SEED}')) AS band_rank
      FROM core c JOIN selected_users u USING(user_id) GROUP BY c.user_id, u.activity_band
    ), interleaved AS (
      SELECT *, sum(degree) OVER (ORDER BY band_rank, activity_band ROWS UNBOUNDED PRECEDING) AS cumulative_rows
      FROM band_ranked
    ) SELECT user_id FROM interleaved WHERE cumulative_rows <= {TARGET_INTERACTIONS}
    """)
    con.execute('CREATE OR REPLACE TABLE core AS SELECT c.* FROM core c JOIN budget_users USING(user_id)')
    iterative_kcore()
print(con.execute('SELECT count(*) rows, count(DISTINCT user_id) users, count(DISTINCT item_id) items FROM core').fetchdf())


## 4. Metadata, kiểm tra coverage, chronological split

Chỉ sau khi chốt item mới quét metadata. Dataset cuối loại item thiếu title; image có thể thiếu và sẽ được xử lý bằng missing-modality policy ở giai đoạn embedding. Sau join, K-core được chạy lại.

In [ ]:
mp = meta_path.as_posix().replace("'", "''")
con.execute(f"""
CREATE OR REPLACE TABLE items AS
SELECT m.parent_asin::VARCHAR item_id, m.title::VARCHAR title,
       array_to_string(m.features, ' ') features, array_to_string(m.description, ' ') description,
       m.main_category::VARCHAR category, m.store::VARCHAR brand, try_cast(m.price AS DOUBLE) price,
       coalesce(m.images[1].large, m.images[1].hi_res, m.images[1].thumb)::VARCHAR image_url
FROM read_json_auto('{mp}', format='newline_delimited', compression='gzip', maximum_object_size=33554432,
                    ignore_errors=true) m
JOIN (SELECT DISTINCT item_id FROM core) wanted ON m.parent_asin = wanted.item_id
QUALIFY row_number() OVER (PARTITION BY m.parent_asin ORDER BY m.title IS NOT NULL DESC) = 1
""")
coverage = con.execute('SELECT count(*) wanted, count(i.item_id) matched, count(i.image_url) with_image FROM (SELECT DISTINCT item_id FROM core) c LEFT JOIN items i USING(item_id)').fetchdf()
print(coverage)
con.execute("CREATE OR REPLACE TABLE core AS SELECT c.* FROM core c JOIN items i USING(item_id) WHERE nullif(trim(i.title), '') IS NOT NULL")
iterative_kcore()
con.execute("""
CREATE OR REPLACE TABLE split AS
SELECT *, CASE row_number() OVER (PARTITION BY user_id ORDER BY timestamp DESC, item_id)
  WHEN 1 THEN 'test' WHEN 2 THEN 'valid' ELSE 'train' END AS split
FROM core
""")
print(con.execute('SELECT split, count(*) rows, count(DISTINCT user_id) users, count(DISTINCT item_id) items FROM split GROUP BY 1 ORDER BY 1').fetchdf())


## 5. Validation, export và manifest

Các kiểm tra dưới đây fail-fast nếu split rò rỉ thời gian, user thiếu partition, hoặc output vượt target quá xa. Parquet dùng ZSTD để thuận tiện tải về.

In [ ]:
checks = {}
checks['users_without_all_splits'] = con.execute("SELECT count(*) FROM (SELECT user_id FROM split GROUP BY 1 HAVING count(DISTINCT split) <> 3)").fetchone()[0]
checks['temporal_leakage'] = con.execute("""
SELECT count(*) FROM (SELECT user_id, max(timestamp) FILTER (WHERE split='train') train_max,
 min(timestamp) FILTER (WHERE split='valid') valid_ts, min(timestamp) FILTER (WHERE split='test') test_ts
 FROM split GROUP BY user_id) WHERE train_max > valid_ts OR valid_ts > test_ts
""").fetchone()[0]
checks['duplicate_user_item'] = con.execute('SELECT count(*)-count(DISTINCT (user_id,item_id)) FROM split').fetchone()[0]
assert all(v == 0 for v in checks.values()), checks

for name in ['train', 'valid', 'test']:
    out = (WORK / f'{name}.parquet').as_posix()
    con.execute(f"COPY (SELECT user_id,item_id,rating,timestamp,verified_purchase FROM split WHERE split='{name}' ORDER BY user_id,timestamp) TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
con.execute(f"COPY (SELECT i.* FROM items i JOIN (SELECT DISTINCT item_id FROM core) c USING(item_id) ORDER BY item_id) TO '{(WORK/'items.parquet').as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")

counts = con.execute('SELECT split, count(*) rows, count(DISTINCT user_id) users, count(DISTINCT item_id) items FROM split GROUP BY 1 ORDER BY 1').fetchdf().to_dict('records')
params = dict(target_users=TARGET_USERS, target_interactions=TARGET_INTERACTIONS, min_user_degree=MIN_USER_DEGREE,
              min_item_degree=MIN_ITEM_DEGREE, positive_rating=POSITIVE_RATING, selection_seed=SELECTION_SEED,
              selection='activity-stratified stable-hash user sampling; full histories; iterative k-core; chronological LLO')
manifest = {'dataset': 'Amazon Reviews 2023 / Clothing_Shoes_and_Jewelry', 'parameters': params,
            'counts': counts, 'checks': checks, 'metadata_coverage': coverage.to_dict('records'),
            'source_files': {'reviews': review_path.name, 'metadata': meta_path.name}}
manifest_text = json.dumps(manifest, ensure_ascii=False, indent=2)
manifest['manifest_sha256'] = hashlib.sha256(manifest_text.encode()).hexdigest()
(WORK / 'dataset_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
con.close()
print(json.dumps(manifest, ensure_ascii=False, indent=2))


In [ ]:
# Zip chỉ các artifact cần tải về; không kèm raw và DuckDB trung gian.
# Tạo archive gọn bằng whitelist, không nén raw/DB trung gian.
subprocess.run(['zip', '-j', '-9', '/kaggle/working/datn_smart_dataset.zip',
                *[str(WORK / x) for x in ['train.parquet','valid.parquet','test.parquet','items.parquet','dataset_manifest.json']]], check=True)
print('Tải file:', '/kaggle/working/datn_smart_dataset.zip', f'({Path("/kaggle/working/datn_smart_dataset.zip").stat().st_size/1024**2:.1f} MiB)')


## Vì sao cách cắt này hợp lý?

- Sampling theo **user** giữ nguyên chuỗi lịch sử cần cho collaborative filtering và chronological split.
- Activity strata giảm thiên lệch do chỉ chọn heavy users nhưng vẫn bảo đảm mỗi user đủ lịch sử.
- Stable hash làm lựa chọn độc lập với thứ tự file, tái lập được và không cherry-pick theo rating/item.
- Iterative K-core tạo đồ thị đủ đặc để BPR/MF học được; chạy lại sau metadata join và sau giảm ngân sách.
- Không ép item đạt target bằng random row sampling; số item là hệ quả tự nhiên của history đã chọn. Điều này đúng hơn cho bài toán Top-K.
- Test/validation là hai tương tác cuối của mỗi user, nên không rò rỉ tương lai như random 80/20.

Không dùng subset này để tuyên bố thống kê về toàn bộ Amazon. Nó được thiết kế cho thí nghiệm recommendation có giới hạn tài nguyên; manifest phải đi cùng mọi kết quả.